# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [518]:
#Import necessary libraries
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import anthropic
import bs4
from IPython.display import Markdown, display, update_display
import requests
import json
import datetime

In [519]:
#load the .env file
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
ipgeo_api_key = os.getenv('IPGEO_API_KEY')

In [520]:
system_message = f'You are a helpful assistant.  If you do not know the answer to a question, please say so.  The current date is {datetime.datetime.today().strftime('%Y-%m-%d %H:%M:%S')}. You have access to tools but only use them if applicable. Response normally otherwise. Respond in Markdown.'
#print(system_message)
openai = OpenAI()
claude = anthropic.Anthropic()
history = []

In [521]:
def get_lat_long_by_ip():
    #Get the lat/Long by IP address.
    lat_long = {}
    url = f'https://api.ipgeolocation.io/v2/ipgeo?apiKey={ipgeo_api_key}'
    request = requests.get(url)
    content = bs4.BeautifulSoup(request.content, "html.parser")
    content_json = json.loads(content.text)
    lat_long.update({'lat' : content_json['location']['latitude']})
    lat_long.update({'long' : content_json['location']['longitude']})

    return lat_long

def get_weather_forecast(date) -> str:
    """Get the weather forecast for a specific date"""
    try:
        # Get the lat/long value by IP
        lat_long = get_lat_long_by_ip()
        lat = lat_long.get('lat')
        long = lat_long.get('long')
        
        if not lat or not long:
            return "Error: Could not determine location from IP address"
        
        # Use the lat/long to get the forecast URL from the National Weather Service
        weather_url = f'https://api.weather.gov/points/{lat},{long}'
        request = requests.get(weather_url)
        
        # Check if the request was successful
        if request.status_code != 200:
            return f"Error: Unable to get weather data. Status code: {request.status_code}"
        
        # Check if response content is not empty
        if not request.content:
            return "Error: Empty response from weather service"
            
        try:
            content = request.json()  # Use .json() method instead of manual parsing
        except json.JSONDecodeError:
            return "Error: Invalid JSON response from weather service"
        
        # Check if the expected data structure exists
        if 'properties' not in content or 'forecast' not in content['properties']:
            return "Error: Unexpected response format from weather service"
            
        forecast_url = content['properties']['forecast']
        
        # Call the forecast URL and scrape out the forecast periods into a dictionary
        forecast_request = requests.get(forecast_url)
        
        if forecast_request.status_code != 200:
            return f"Error: Unable to get forecast data. Status code: {forecast_request.status_code}"
            
        if not forecast_request.content:
            return "Error: Empty forecast response"
            
        try:
            forecast = forecast_request.json()  # Use .json() method
        except json.JSONDecodeError:
            return "Error: Invalid JSON in forecast response"
        
        # Check forecast structure
        if 'properties' not in forecast or 'periods' not in forecast['properties']:
            return "Error: Unexpected forecast response format"
            
        forecast_periods = {}
        for p in forecast['properties']['periods']:
            if 'startTime' in p and 'detailedForecast' in p:
                forecast_periods[str(p['startTime'])[:10]] = p['detailedForecast']

        result = forecast_periods.get(date)
        if result:
            return result
        else:
            # If exact date not found, return available dates
            available_dates = list(forecast_periods.keys())
            return f"No forecast available for {date}. Available dates: {', '.join(available_dates)}"
            
    except Exception as e:
        return f'Error getting weather forecast: {str(e)}'

In [522]:
weather_tool_openai = {
    "name": "get_weather_forecast",
    "description": "Get the weather forecast for a specific date, use this to determine an answer to questions like 'will it rain tomorrow?'",
    "parameters": {
        "type": "object",
        "properties": {
            "date": {
                "type": "string",
                "description": "The forecast date",
            },
        },
        "required": ["date"]
    }
}

weather_tool_claude = {
    "type": "custom",
    "name": "get_weather_forecast",
    "description": "Get the weather forecast for a specific date, use this to determine an answer to questions like 'will it rain tomorrow?'",
    "input_schema": {
        "type": "object",
        "properties": {
            "date": {
                "type": "string",
                "description": "The forecast date",
            },
        },
        "required": ["date"]
    }
}

In [523]:
tools_openai = [{"type": "function", "function": weather_tool_openai}]
tools_claude = [weather_tool_claude]

In [524]:
# HANDLE THE TOOL CALL FROM THE LLM - WITH DEBUGGING
def handle_tool_call(tool_call_dict):
    #print(f"DEBUG: handle_tool_call received: {tool_call_dict}")
    
    try:
        # Extract function info
        if 'function' not in tool_call_dict:
            error_msg = "Error: No 'function' key in tool call"
            #print(f"DEBUG: {error_msg}")
            return {
                "role": "tool",
                "tool_call_id": tool_call_dict.get('id', 'unknown'),
                "content": json.dumps({"error": error_msg})
            }
        
        function_name = tool_call_dict['function'].get('name', '')
        raw_args = tool_call_dict['function'].get('arguments', '')
        tool_call_id = tool_call_dict.get('id', 'unknown')
        
        #print(f"DEBUG: Function: {function_name}, Args: '{raw_args}', ID: {tool_call_id}")
        
        # Validate arguments
        if not raw_args or raw_args.strip() == '':
            error_msg = f"Error: Empty arguments for function {function_name}"
            #print(f"DEBUG: {error_msg}")
            return {
                "role": "tool",
                "tool_call_id": tool_call_id,
                "content": json.dumps({"error": error_msg})
            }
        
        # Parse arguments
        try:
            arguments = json.loads(raw_args)
            #print(f"DEBUG: Parsed arguments: {arguments}")
        except json.JSONDecodeError as e:
            error_msg = f"Error parsing JSON arguments: {e}"
            #print(f"DEBUG: {error_msg}")
            return {
                "role": "tool",
                "tool_call_id": tool_call_id,
                "content": json.dumps({"error": error_msg})
            }
        
        # Handle specific function calls
        if function_name == 'get_weather_forecast':
            date = arguments.get('date')
            if not date:
                error_msg = "Error: No date provided in arguments"
                #print(f"DEBUG: {error_msg}")
                return {
                    "role": "tool",
                    "tool_call_id": tool_call_id,
                    "content": json.dumps({"error": error_msg})
                }
            
            #print(f"DEBUG: Calling get_weather_forecast with date: {date}")
            forecast = get_weather_forecast(date)
            #print(f"DEBUG: Weather forecast result: {forecast}")
            
            return {
                "role": "tool",
                "tool_call_id": tool_call_id,
                "content": json.dumps({"forecast": forecast})
            }
        else:
            error_msg = f"Error: Unknown function {function_name}"
            #print(f"DEBUG: {error_msg}")
            return {
                "role": "tool",
                "tool_call_id": tool_call_id,
                "content": json.dumps({"error": error_msg})
            }
            
    except Exception as e:
        error_msg = f"Error in handle_tool_call: {str(e)}"
        #print(f"DEBUG: {error_msg}")
        #import traceback
        #print(f"DEBUG: Traceback: {traceback.format_exc()}")
        return {
            "role": "tool",
            "tool_call_id": tool_call_dict.get('id', 'unknown'),
            "content": json.dumps({"error": error_msg})
        }

In [525]:
# OPENAI CHAT
def chat_openai(message, history):
    try:
        messages = [{"role": "system", "content": system_message}]
        messages.extend(history)
        messages.append({"role": "user", "content": message})
        #print(f"DEBUG: Sending messages to OpenAI: {messages}")
    
        stream = openai.chat.completions.create(
            model='gpt-4o-mini',
            messages=messages,
            tools=tools_openai,
            stream=True
        )
    
        response = ""
        tool_call_chunks = {}
        tool_call_detected = False
        
        # Generator function to yield content and update history
        chunk_id = ""
        for chunk in stream:
            #print(f"DEBUG: Raw chunk: {chunk}")
            choice = chunk.choices[0]
            #print(f"DEBUG: Choice: {choice}")
            #print(f"DEBUG: Choice.delta: {choice.delta}")
            
            # Handle tool calls
            if choice.delta.tool_calls:
                #print(f"DEBUG: Tool call chunk received: {choice.delta.tool_calls}")
                tool_call_detected = True
                
                for tc in choice.delta.tool_calls:
                    #print(f"DEBUG: Raw tool call object: {tc}")
                    #print(f"DEBUG: Tool call ID: {tc.id}")
                    #print(f"DEBUG: Tool call type: {tc.type}")
                    #print(f"DEBUG: Tool call function: {tc.function}")
                    
                    #if tc.function:
                    #    print(f"DEBUG: Function name: {repr(tc.function.name)}")
                    #    print(f"DEBUG: Function arguments: {repr(tc.function.arguments)}")
                    #    print(f"DEBUG: Function arguments type: {type(tc.function.arguments)}")
                    
                    # Initialize tool call if not exists
                    #chunk_id = ""
                    if tc.id is not None:
                        chunk_id = tc.id
                    #print(f'DEBUG: Chunk id is {chunk_id} and tc.id is {tc.id}')
                    if tc.id not in tool_call_chunks:
                        tool_call_chunks[tc.id] = {
                            "id": chunk_id,
                            "function": {
                                "name": "",
                                "arguments": ""
                            },
                            "type": tc.type or "function"
                        }
                        #print(f"DEBUG: Initialized new tool call chunk for ID: {chunk_id}")
                        
                    # Accumulate function name
                    if tc.function and tc.function.name is not None:
                        tool_call_chunks[chunk_id]["function"]["name"] += tc.function.name
                        #print(f"DEBUG: Function name chunk: id = {chunk_id}, fn = {repr(tc.function.name)}, total so far: {repr(tool_call_chunks[tc.id]['function']['name'])}")
                    
                    # Accumulate function arguments - this is the key fix
                    if tc.function and tc.function.arguments is not None:
                        #print(f"DEBUG: Adding arguments chunk: id = {chunk_id}, args = {repr(tc.function.arguments)} (length: {len(tc.function.arguments)})")
                        tool_call_chunks[chunk_id]["function"]["arguments"] += tc.function.arguments
                        #print(f"DEBUG: Arguments total so far: {repr(tool_call_chunks[chunk_id]['function']['arguments'])} (length: {len(tool_call_chunks[chunk_id]['function']['arguments'])})")
                    #else:
                    #    print(f"DEBUG: No arguments in this chunk - tc.function: {tc.function}")
                    #    if tc.function:
                    #        print(f"DEBUG: tc.function.arguments is: {repr(tc.function.arguments)}")
                    
                    #print(f"DEBUG: Current tool_call_chunks state: {tool_call_chunks}")
            
            # Handle regular content
            elif choice.delta.content:
                delta = choice.delta.content
                response += delta
                if delta:
                    yield response  # stream to Gradio frontend
            
            # Check if this is the end of the stream
            #if choice.finish_reason:
            #    print(f"DEBUG: Stream finished with reason: {choice.finish_reason}")
                
        # END FOR CHUNK IN STREAM LOOP
        
        # Update the history list
        history.append({"role": "user", "content": message})

        #print(f"DEBUG: Response is: {response}")
        
        # If no tool call was detected, just add the response and return
        if not tool_call_detected:
            #print(f'DEBUG: Tool call not detected')
            if response:
                history.append({"role": "assistant", "content": response})
            return
    
        # Tool call handling (post-stream)  
        if tool_call_detected:
            #print(f"DEBUG: Final tool_call_chunks: {tool_call_chunks}")
            
            if not tool_call_chunks:
                yield "Error: Tool call detected but no tool calls found"
                return
            
            # Convert tool_call_chunks to list for assistant message
            tool_calls_list = []
            tool_responses_list = []

            #print (f'DEBUG: Looping through tool_call_chunks:\n{tool_call_chunks}\n')
            
            # Process each tool call
            for tool_call_id, tool_call in tool_call_chunks.items():
                if not(tool_call['function']['name'] is None or (tool_call['function']['name']).strip() == ""):
                    #print(f"DEBUG: Processing tool call: {tool_call}")
                    
                    # Validate tool call completeness
                    function_name = tool_call["function"]["name"]
                    function_args = tool_call["function"]["arguments"]
    
                    #print(f'DEBUG: function_args are {function_args}')
                    
                    if not function_name:
                        yield f"Error: Tool call missing function name"
                        return
                        
                    #if not function_args or function_args.strip() == "":
                        # Fallback: Try non-streaming approach
                        #print(f"DEBUG: Arguments empty, trying non-streaming fallback...")
                    
                    # Validate JSON arguments
                    try:
                        json.loads(function_args)
                    except json.JSONDecodeError as e:
                        yield f"Error: Invalid JSON in tool call arguments: {e}"
                        return
                    
                    # Add to tool calls list
                    tool_calls_list.append(tool_call)
                    print(f'DEBUG: Tool call is {tool_call}')
                    
                    # Handle the tool call
                    tool_response = handle_tool_call(tool_call)
                    #print(f"DEBUG: Tool response: {tool_response}")
                    #print(f'DEBUG: JSON tool response content: {tool_response['content']}')

                    #Add to tool responses list
                    tool_responses_list.append({
                        "role":"tool",
                        "tool_call_id":tool_response['tool_call_id'],
                        "content": json.dumps({'forecast': json.loads(tool_response['content'])})
                    })
                    
                    #print(f'DEBUG: Tool response list is {tool_responses_list}')
                # END IF FUNCTION NAME TEST

            # END TOOL_CALLS FOR LOOP
            # Add tool response to messages
            if tool_calls_list:
                messages.append({
                    "role": "assistant", 
                    "content": None,  # Important: set to None when there are tool calls
                    "tool_calls": tool_calls_list
                })

            if tool_responses_list:
                messages.extend(tool_responses_list)

        #print(f"DEBUG: Messages before final completion: {messages}")
    
        # Get final response from assistant
        final_completion = openai.chat.completions.create(
            model='gpt-4o-mini',
            messages=messages,
            temperature=0.7
        )
        
        final_response = final_completion.choices[0].message.content
        #print(f"DEBUG: Final response: {final_response}")
        
        if final_response:
            yield final_response
            history.append({"role": "assistant", "content": final_response})
            
    except Exception as e:
        import traceback
        error_msg = f'Error in chat_openai: {str(e)}\n{traceback.format_exc()}'
        print(error_msg)
        yield error_msg

In [534]:
def chat_claude(message, history):
    result = claude.messages.stream(
        model="claude-opus-4-20250514",
        max_tokens = 1000,
        temperature = 0.4,
        system = system_message,
        tools = tools_claude,
        messages = history + [{"role":"user","content":message}]
    )
    response = ""
    tool_calls = []
    tool_responses = []
    tool_call_id = ""
    tool_call_fn = ""
    tool_call_args = ""
    tool_call_detected = False
    tool_call_item = None
    
    with result as stream:
        for chunk in stream:
            if chunk.type == 'content_block_start' and chunk.content_block.type == 'tool_use':
                tool_call_detected = True
                tool_call_id = chunk.content_block.id
                tool_call_fn = chunk.content_block.name
                tool_call_item = {
                        "id":chunk.content_block.id,
                        "function":{
                            "name":chunk.content_block.name,
                            "arguments":""
                        },
                        "type":"function"
                    }
                print(f'DEBUG: Claude detected a tool call : {chunk}')
                print(f'DEBUG: Tool call item definition is {tool_call_item}')

                response += f'...using tool {tool_call_fn}...'
                yield response

            elif chunk.type == 'content_block_delta':
                if hasattr(chunk.delta, 'text'):
                    text = chunk.delta.text
                    response += text
                    yield response
                # GET THE TOOL ARGUMENTS IF ANY
                elif hasattr(chunk.delta, 'partial_json') and tool_call_item:
                    tool_call_item['function']['arguments'] += chunk.delta.partial_json
            elif chunk.type == 'content_block_stop' and tool_call_detected:
                if hasattr(chunk, 'content_block') and chunk.content_block.type == 'tool_use':
                    print(f'DEBUG: Detected message_top event and tool_call_detected is True')
                    print(f'DEBUG: Chunk is {chunk}')
                    print(f'DEBUG: Content block is {chunk.content_block}')

                    if tool_call_item:
                        try:
                            tool_calls.append(tool_call_item)
                            print(f'DEBUG: Calling tool {tool_call_fn}')
                            tool_response = handle_tool_call(tool_call_item)
                            tool_responses.append({
                                "tool_call_id":tool_call_item['id'],
                                "role":"tool",
                                "content":tool_response
                            })
                            print(f'DEBUG: Tool response is - {tool_response}')

                            response += tool_response
                            yield response
                        except Exception as e:
                            print(f'ERROR: {str(e)}')
                            err_msg = f'ERROR: Tool error - {str(e)}'
                            response += err_msg
                            yield response

                        # Reset for next potential tool call
                        tool_call_item = None
                        tool_call_detected = False
                    
            print(f'DEBUG: Current chunk - {chunk}')
            #print(f'DEBUG: Current response - {response}')
      
            print(f'DEBUG: Total response is - {response}')
            print(f'DEBUG: All tool calls - {tool_calls}')
            print(f'DEBUG: All tool responses - {tool_responses}')

    if tool_response:
        print(f'DEBUG: Making followup request with tool response')

        followup_messages = history + [
            {"role":"user", "content":message},
            {"role":"assistant", "content":response}
        ] + tool_responses

        follow_up_result = claude.messages.stream(
                model="claude-opus-4-20250514",
                max_tokens=1000,
                temperature=0.4,
                system=system_message,
                tools=tools_claude,
                messages=follow_up_messages
            )
            
        response += "\n\n"
        yield response
        
        with follow_up_result as follow_stream:
            for chunk in follow_stream:
                if chunk.type == 'content_block_delta' and hasattr(chunk.delta, 'text'):
                    text = chunk.delta.text
                    response += text
                    yield response
    
    history.append({"role":"user", "content":message})
    history.append({"role":"assistant", "content":response})

    yield response

In [535]:
def stream_chat(message, model):
    if model == 'OpenAI':
        yield from chat_openai(message, history)
    if model == 'Claude':
        yield from chat_claude(message, history)
    if not(model == 'OpenAI' or model == 'Claude'):
        raise ValueError('Sorry we don''t support that model yet!')

In [536]:
#for r in stream_chat('what is the forecast for the next 3 days?', 'Claude'):
#    print(r)

In [537]:
view = gr.Interface(
    fn=stream_chat,
    inputs=[gr.Textbox(label="Message:"), gr.Dropdown(["OpenAI","Claude"], label="Model:")],
    outputs=[gr.Markdown(label="Response:")],
    flagging_mode="never"
)
view.launch()
#gr.ChatInterface(fn=chat_openai, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
